In [ ]:
import pandas as pd 
import re 
import numpy as np 



## step 4 , 

In [ ]:
raw = pd.read_csv('messy_cancer_lab_dataset.csv')
data = raw.copy()      

data.head()
data.info
data.isnull().sum()
data.duplicated().sum()
data.columns
data.info()



<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   order_id                 9484 non-null   str  
 1   signout_date             9506 non-null   str  
 2   lab_number               9502 non-null   str  
 3   assigned_to_pathologist  8570 non-null   str  
 4   patient_name             9451 non-null   str  
 5   facility                 8736 non-null   str  
 6   test_category            10000 non-null  str  
 7   test                     9500 non-null   str  
 8   sub_category             8911 non-null   str  
 9   receiving_centers        8281 non-null   str  
 10  processing_centers       8390 non-null   str  
 11  creation_date            9756 non-null   str  
 12  service                  8499 non-null   str  
 13  price                    9303 non-null   str  
 14  payment_method           8603 non-null   str  
dtypes: str(15)
mem

In [ ]:
data.head(30)
data = data.apply(lambda col: col.str.strip())

data = data.replace(r'(?i)^(unknown|nan)', np.nan, regex=True)

data.head(30)
data.isnull().sum()

order_id                    516
signout_date                494
lab_number                  498
assigned_to_pathologist    2360
patient_name                586
facility                   2564
test_category               994
test                       3635
sub_category               1089
receiving_centers          3377
processing_centers         3274
creation_date               244
service                    2898
price                       697
payment_method             2866
dtype: int64

In [ ]:
#fix the date 
data['creation_date']

data['signout_date'] = pd.to_datetime(data['signout_date'], errors='coerce', format= 'mixed')
data['creation_date'] = pd.to_datetime(data['creation_date'], errors='coerce', format= 'mixed')

data[["signout_date", 'creation_date']]



,signout_date,creation_date
0,2021-04-02,2022-05-09
1,2021-03-16,2023-06-20
2,2022-02-17,2025-03-10
3,2021-03-08,2025-01-12
4,2020-11-23,2022-06-09
...,...,...
9995,2024-03-28,2022-10-12
9996,2023-11-17,2023-09-27
9997,2022-01-24,2021-11-11
9998,2021-01-23,NaT


In [ ]:
# fix the price 

data['price']
data['price'] = data['price'].str.strip().str.replace(r'[^0-9]', '', regex=True)
data['price'] = pd.to_numeric(data['price'])
data['price'] = data['price'].replace(0, np.nan)
data['price'].value_counts()


price
2625.0    6
1213.0    6
5016.0    6
5326.0    6
7437.0    6
         ..
6588.0    1
2083.0    1
968.0     1
7216.0    1
3811.0    1
Name: count, Length: 5940, dtype: int64

In [ ]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   order_id                 9484 non-null   str           
 1   signout_date             9030 non-null   datetime64[us]
 2   lab_number               9502 non-null   str           
 3   assigned_to_pathologist  7640 non-null   str           
 4   patient_name             9414 non-null   str           
 5   facility                 7436 non-null   str           
 6   test_category            9006 non-null   str           
 7   test                     6365 non-null   str           
 8   sub_category             8911 non-null   str           
 9   receiving_centers        6623 non-null   str           
 10  processing_centers       6726 non-null   str           
 11  creation_date            9493 non-null   datetime64[us]
 12  service                  7102 non-null   str

In [ ]:
data['processing_centers'] = data['processing_centers'].str.title()
data['processing_centers']

0               NaN
1          Main Lab
2       Central Lab
3               NaN
4          Main Lab
           ...     
9995            NaN
9996    Central Lab
9997            NaN
9998    Central Lab
9999            NaN
Name: processing_centers, Length: 10000, dtype: str

In [ ]:
data['service'] = data['service'].str.title()

In [ ]:

map_payment_method = {'credit': "Credit Card" , 'ins': "Insurance"}
data['payment_method'] = data['payment_method'].str.replace(map_payment_method)
data['payment_method']



0               NaN
1               NaN
2              Cash
3       Credit Card
4               NaN
           ...     
9995    Credit Card
9996            NaN
9997    Credit Card
9998            NaN
9999            NaN
Name: payment_method, Length: 10000, dtype: str

In [ ]:
data['receiving_centers'] = data['receiving_centers'].str.title().unique()
data['receiving_centers']

ValueError: Length of values (4) does not match length of index (10000)

In [ ]:
data['test'] = data['test'].unique()

<StringArray>
[                     nan,          'ER/PR Testing',           'HER2 Testing',
         'Cytology Smear', 'Fine Needle Aspiration', 'Bone Marrow Aspiration',
        'Tissue Staining',                 'Biopsy',            'Blood Count']
Length: 9, dtype: str

In [ ]:
data['facility'] = data['facility'].str.title().value_counts()

facility
Mercy Cancer Center    2500
St. Marys              1284
General Hospital       1247
City Cancer Clinic     1203
St. Mary'S Oncology    1202
Name: count, dtype: int64

In [ ]:
#assigned to pathologist 
data['assigned_to_pathologist'] = data['assigned_to_pathologist'].str.title().str.replace(r'(?i)^dr\.?\s*', "", regex=True)
data['assigned_to_pathologist'] = data['assigned_to_pathologist'].replace('Smyth', 'Smith')
doctor = data['assigned_to_pathologist']
data['assigned_to_pathologist'] = 'Dr. ' + doctor
data['assigned_to_pathologist']



0               Dr. Smith
1            Dr. Jane Doe
2        Dr. Justin Smith
3               Dr. Smith
4            Dr. John Lee
              ...        
9995    Dr. Julia Pearson
9996                  NaN
9997            Dr. Smith
9998         Dr. Jane Doe
9999            Dr. Smith
Name: assigned_to_pathologist, Length: 10000, dtype: str

In [ ]:
data['test_category'] = data['test_category'].unique()


<StringArray>
[     'Immunohistochem',                    nan,          'Haematology',
             'Cytology',            'histology', 'Immunohistochemistry',
            'Histology',             'CYTOLOGY',           'Hematology',
                  'IHC']
Length: 10, dtype: str

In [ ]:
#sub category standardization 
category_mapping = {
    'Immunohistochem': 'Immunohistochemistry',
    'IHC': 'Immunohistochemistry',
    'Haematology': 'Hematology',
    'Hematology': 'Hematology',
    'CYTOLOGY': 'Cytology',
    'Cytology': 'Cytology',
    'histology': 'Histology',
    'Histology': 'Histology'
}

data['test_category'] = data['test_category'].str.title().replace(category_mapping)

data['test_category']

0       Immunohistochemistry
1       Immunohistochemistry
2                        NaN
3                 Hematology
4                   Cytology
                ...         
9995                Cytology
9996              Hematology
9997              Hematology
9998                Cytology
9999                Cytology
Name: test_category, Length: 10000, dtype: str

In [ ]:
# data[data['order_id'].duplicated(keep=False)]
repeated = data[
    data["order_id"].notna() &
    data["order_id"].duplicated(keep=False)
]
repeated[["order_id", "lab_number", "patient_name", "signout_date"]].sort_values("order_id")


#  duplicated row has same order id but different lab number and patient name ...so its maybe order_id mistake ...so we need to flag it to further 
# investigation  

data['repeted_order_id_need_investigation'] = data['order_id'].notnull() & data['order_id'].duplicated(keep=False) 
data.duplicated().sum()

np.int64(0)

In [ ]:
data[data['order_id'].isnull()]

,order_id,signout_date,lab_number,assigned_to_pathologist,patient_name,facility,test_category,test,sub_category,receiving_centers,processing_centers,creation_date,service,price,payment_method
0,NaN,2021-04-02,LAB81482,Dr. Smith,William Evans,St. Mary's Oncology,Immunohistochem,NaN,Leukemia,NaN,NaN,2022-05-09,Routine,206.0,NaN
16,NaN,2022-11-30,LAB97900,Dr. Smyth,Morgan Serrano,City Cancer Clinic,Cytology,Cytology Smear,Lung CA,Main Lab,NaN,2022-04-04,Stat,6311.0,NaN
63,NaN,2022-02-03,LAB18855,Dr. Smyth,Tracy Hunter,City Cancer Clinic,Haematology,NaN,Lung CA,Main Lab,NaN,2024-05-14,Routine,3086.0,NaN
77,NaN,2020-11-06,LAB90147,Dr. Smyth,Mark Smith,mercy cancer center,Immunohistochem,ER/PR Testing,NaN,NaN,Central Lab,2020-12-20,Consult,4431.0,Credit Card
103,NaN,2022-11-11,LAB85397,NaN,Peter Evans,mercy cancer center,CYTOLOGY,Cytology Smear,Lung Cancer,Main Lab,Central Lab,2020-06-24,NaN,5210.0,Credit Card
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9885,NaN,2023-06-11,LAB46968,Dr. Smith,Alison Young,mercy cancer center,Histology,Biopsy,Leukemia,NaN,NaN,2025-04-13,Urgent,2977.0,INS
9914,NaN,2023-07-12,LAB98343,Dr. Smith,Chris Long,mercy cancer center,Histology,Tissue Staining,Lymphoma,lab a,Central Lab,2022-12-05,Routine,4745.0,Cash
9959,NaN,2021-05-12,LAB83833,Dr. Smith,Gilbert Harris,mercy cancer center,IHC,HER2 Testing,NS,Lab B,Main Lab,2021-12-26,Consult,2999.0,NaN
9969,NaN,2020-01-23,LAB27581,Dr. Smith,Justin Young,St. Marys,Cytology,NaN,Breast Cancer,Main Lab,Lab C,2021-03-24,Urgent,4825.0,Insurance


In [ ]:
#final check and save 
data.to_csv('messy_cancer_lab_dataset_cleaned_version.csv', index=False)


,order_id,signout_date,lab_number,assigned_to_pathologist,patient_name,facility,test_category,test,sub_category,receiving_centers,processing_centers,creation_date,service,price,payment_method
0,NaN,2021-04-02,LAB81482,Dr. Smith,William Evans,St. Mary's Oncology,Immunohistochem,unknown,Leukemia,NaN,NaN,2022-05-09,routine,206,NaN
1,ORD6574,03/16/2021,LAB57052,Dr Jane Doe,NaN,NaN,Immunohistochem,ER/PR Testing,Lung Cancer,Unknown,Main Lab,06/20/2023,Consult,ksh 3833,NaN
2,ORD4814,02/17/2022,LAB31319,Justin Smith,Lori Diaz,General Hospital,Unknown,unknown,Leukemia,Lab B,central lab,10-Mar-25,Unknown,1016,Cash
3,ORD7572,08-Mar-21,LAB95909,Dr. Smyth,Juan Carrillo,St. Marys,Haematology,unknown,Lung CA,Main Lab,NaN,12-Jan-25,routine,2366,credit
4,ORD2796,23-Nov-20,LAB60019,Dr. John Lee,Joanna Tyler,Mercy Cancer Center,Cytology,unknown,Not Specified,NaN,Main Lab,2022-06-09,routine,ksh 0,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,ORD6349,28-Mar-24,LAB61105,Julia Pearson,Mark Warner,St. Mary's Oncology,Cytology,Fine Needle Aspiration,Not Specified,Main Lab,Unknown,12-Oct-22,Urgent,166,credit
9996,ORD7563,11/17/2023,LAB59550,NaN,Amanda Ward,NaN,Haematology,unknown,Breast Cancer,Main Lab,central lab,27-Sep-23,routine,ksh 3592,NaN
9997,ORD8416,01/24/2022,LAB58490,Dr. Smyth,Danielle Rowe,St. Marys,Hematology,Blood Count,Leukemia,Main Lab,Unknown,11/11/2021,Unknown,2018,Credit Card
9998,ORD8883,01/23/2021,LAB79062,Dr. Jane Doe,Shawna Harrison,NaN,Cytology,Fine Needle Aspiration,Lymphoma,Lab A,Central Lab,invalid,STAT,NaN,NaN
